In [1]:
# from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
# from rouge_score import rouge_scorer
from pathlib import Path
import sys
import json
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig
from peft import PeftModel, PeftConfig
import torch
import gc

2025-04-01 21:36:55.389285: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
gc.collect()
model_name = "meta-llama/Llama-3.2-3B-Instruct"  # or the base model you trained on
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
config = AutoConfig.from_pretrained(model_name)
# manually set rope_scaling to supported structure:
config.rope_scaling = {"type": "dynamic", "factor": 2.0}
config.use_cache = True
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    config=config,
    torch_dtype=torch.float16
)


adapter_path = "./../Training/final_adapter_with_eval_1"  # or wherever your adapter_model.safetensors is
#adapted_model= PeftModel.from_pretrained(base_model,adapter_path)
lora_config = PeftConfig.from_pretrained(adapter_path) 
base_model.add_adapter(lora_config, adapter_name="summarizer")
base_model.disable_adapters()



Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [3]:
import fitz
def extract_normally( file_path: Path, ocr : bool = False): 
    #in_path = os.path.abspath( in_base + relative_file_path)
    in_path = file_path# Maintain folder structure

    try: 

        doc = fitz.open(in_path)  # Open the PDF document
        # Ensure the output directory exists
        text =""
        for page in doc:  # Iterate through pages
            if not ocr:
                text += page.get_text()  # Get page text
            else:
                partial_tp = page.get_textpage_ocr(flags=0, full = True)
                text += page.get_text(textpage=partial_tp)  # Get page text
        # Create output file name (replace .pdf with .txt)
        return text
    except Exception as e:
        # Print the exception type, the error message, and the line number
        tb = sys.exc_info()[2]
        lineno = tb.tb_lineno
        # Print the exception type, the error message, and the line number
        print(f"Exception Type: {type(e).__name__}")
        print(f"Error Message: {e}")
        print(f"Line Number: {lineno}") 

In [4]:
def chunk_text_with_overlap(text, chunk_size=500, overlap=100):
    """Splits text into overlapping chunks."""
    chunks = []
    start = 0
    while start < len(text):
        end_ptr = min(start + chunk_size, len(text))
        chunks.append(text[start:end_ptr])
        start += chunk_size - overlap  # Move forward while keeping overlap

    # for chunk in chunks:
    #     print(chunk)
    return chunks


In [13]:
def clean_summary(text):
    return (
        text.replace('NULL', '')
            .replace('<', '')
            .replace('>', '')
            .replace('\n', '')
            .replace('--', '')
            .replace('Summary:', '')
    )
class RecursiveSummarizer:
    def __init__(self,model):
         self.model = model
     
    def summarize(self, chunks, max_token, chunk_limit):
          if len(chunks) == chunk_limit:
               return " ".join(chunks) # Base case: only one chunk remains

          summarized_chunks = []

          for chunk in chunks:
               summary = self.generate_summary(chunk, adapted_model=self.model, max_new_tokens=max_token)
               cleaned = clean_summary(summary)
               summarized_chunks.append(cleaned)

          # Combine and re-chunk
          combined = " ".join(summarized_chunks)
          combined = combined.strip() 
          print("<----Combined--->", combined)

          # Recursively call after re-chunking
          new_chunks = chunk_text_with_overlap(text=combined, chunk_size=900)
          return self.summarize(new_chunks, max_token, chunk_limit=chunk_limit)

    def generate_summary(self, input_text, adapted_model, max_new_tokens=150):
        torch.cuda.empty_cache()
        prompt = f"""Summarize:\n{input_text} Summary:\n"""

        inputs = tokenizer(prompt, return_tensors="pt").to(adapted_model.device)
    
        with torch.no_grad():
            outputs = adapted_model.generate(
                **inputs,
                do_sample=True,
                temperature=0.7,
                max_new_tokens=max_new_tokens,
                top_p=0.9
            )
        input_len = inputs["input_ids"].shape[1]
        new_tokens = outputs[0][input_len:]  # exclude prompt
        summary = tokenizer.decode(new_tokens, skip_special_tokens=True)
        return summary


##### Make sure to install tesseract, the machine learning component of fitz for ocr

In [14]:
pdf_path = "./Data/2106.04560v1.pdf"
normal_txt = extract_normally(pdf_path,ocr=False)
print(normal_txt)
print("<--------------->")
ocr_txt = extract_normally(pdf_path,ocr=True)
print(ocr_txt)
base_model.set_adapter("summarizer")
summarizer = RecursiveSummarizer(model=base_model)
normal_chunks = chunk_text_with_overlap(normal_txt)
ocr_chunks = chunk_text_with_overlap(ocr_txt)
print(normal_chunks)
print(summarizer.summarize(normal_chunks,max_token=100,chunk_limit=3))
print(summarizer.summarize(ocr_chunks, max_token=100,chunk_limit=3))

Scaling Vision Transformers
Xiaohua Zhai∗, Alexander Kolesnikov∗, Neil Houlsby, Lucas Beyer∗
Google Research, Brain Team, Zürich
{xzhai,akolesnikov,neilhoulsby,lbeyer}@google.com
Abstract
Attention-based neural networks such as the Vision Transformer (ViT) have recently
attained state-of-the-art results on many computer vision benchmarks. Scale is a
primary ingredient in attaining excellent results, therefore, understanding a model’s
scaling properties is a key to designing future generations effectively. While the
laws for scaling Transformer language models have been studied, it is unknown
how Vision Transformers scale. To address this, we scale ViT models and data,
both up and down, and characterize the relationships between error rate, data, and
compute. Along the way, we reﬁne the architecture and training of ViT, reducing
memory consumption and increasing accuracy of the resulting models. As a result,
we successfully train a ViT model with two billion parameters, which attains
a 

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Scaling Vision Transformers
‘inoZa Alexanderoles’ Nel Housy Laas Beer"
Google Resch, Bn Team, ich
a
Abstract
a
a
Anion asener tors schaste VisionTransformer (VThaere
5
‘ined athe rents
onmany computer vonDecne
Z
rinnecting acl ale ert undandn els
2
{Eling oer
ay to digg fu enranseeciney Whee
ioys fi sling Transfer language models hve ben sad
ison
=
tow Vision Tunfarmens tle Toad ths we ale Vt models nd da
5
Toth upand Jv,nd haateize heaos eweenevra at an
5
Compu Along theay eens heactetand aining ofTEeng
)
tremor conunptn and ncening cry ot
esuling mods Aes
g
‘we successfully train a ViT model with two billion parameters,
which attai
&
Yew sutcoftheartontmageNet of 901% tpt tcuncy. The mol a
evo wll on wang
example, each
50 top acc
=
SargeNt wh only
10 eras perce
2
1
Introduction
=
=
Anion sed Transformeratts[35]hvtks comput vison demain bys
rd
‘nfa becoming an ncesig) popular htenese
and rate: revoT
sg
ive boon wie aoe the atl langue poceing LP domain
Opti
S
‘Tranter
EP waaay ses
with

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for

<----Combined---> The Vision Transformer (ViT) has achieved state-of-the-art results on many computer vision benchmarks. However, understanding its scaling properties is crucial to designing future generations of models. The ViT uses self-attention mechanisms, which can lead to a significant increase in model size and computational cost. To address this issue, the authors propose scaling the ViT by using a hierarchical attention mechanism, which allows the model to focus on specific regions of the image while reducing the number of parameters and computations. The proposed Vision Transformers (ViT) are a type of deep learning model that has shown promise in image recognition tasks. However, understanding how these models scale with increasing data and computational resources is crucial for their effective deployment in real-world applications.The authors of the paper investigated the scaling properties of ViT models and data, both up and down, to better understand how they behave under

...

<----Combined---> The paper proposes two new methods for training deep neural networks: Ensemble of Temporally Decorrelated Models (ETDM) and Deep Reinforcement Learning with Autoencoder (DRL-AE). ETDM combines multiple models with different architectures to capture complex behavior, while DRL-AE combines techniques from reinforcement learning and deep learning to optimize neural networks. The paper also explores the relation between the two methods and discusses the potential applications of the proposed methods in various fields such as computer vision, natural language processing. The text appears to be a response to a prompt asking for a summary of an article. However, the provided text does not clearly state the article's title or content. I can provide a summary based on the given information.The text mentions a few themes, including:* A comparison between two systems, JFF-0M and FFB* A self-supervised approach for few-shot learning called WITGVI4* A new method for reducing the c

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


<----Combined---> The paper proposes two new methods for training deep neural networks: Ensemble of Temporally Decorrelated Models (ETDM) and Deep Reinforcement Learning with Autoencoder (DRL-AE). ETDM combines multiple models with different architectures to capture complex behavior, while DRL-AE combines techniques from reinforcement learning and deep learning to optimize neural networks. The paper also explores the relation between the two methods and discusses the potential applications of the proposed methods in various fields such as computer vision, natural language processing. The paper discusses the challenges and trade-offs associated with building and maintaining large-scale datacenters. It presents a comprehensive framework for scalability, cost, and energy efficiency, and provides best practices and guidelines for scalability. The authors emphasize the importance of understanding trade-offs between different design elements, such as power consumption, cooling, and capacity.

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


<----Combined---> The paper proposes two new methods for training deep neural networks: Ensemble of Temporally Decorrelated Models (ETDM) and Deep Reinforcement Learning with Autoencoder (DRL-AE). ETDM combines multiple models with different architectures to capture complex behavior, while DRL-AE combines techniques from reinforcement learning and deep learning to optimize neural networks. The paper also explores the relation between the two methods and discusses potential applications in computer vision, natural language processing. The authors discuss challenges and trade-offs associated with The paper discussed the importance of scalability in datacenter design, emphasizing the need to understand trade-offs between different design elements such as power consumption, cooling, and capacity. The study concluded that scalability is a critical aspect of datacenter design and that understanding these trade-offs is essential for building efficient and cost-effective datacenters. The secon

In [ ]:
The study examined the performance of the Vision Transformer (ViT) model on image classification tasks and proposed a new approach to few-shot learning using hierarchical attention. The results showed that ViT models perform well on image classification tasks, but their error rate increases with the amount of data. The study also found that the learning rate is a bottleneck in terms of representation quality, and that even with infinite capacity, the error rate of a model does not decrease to zero, but instead reaches a saturation point. Additionally The article discusses a new vision transformer architecture called PVT (Parallel Vision Transformer), which is designed to improve the performance of vision transformers in learning features from images. The PVT architecture is robust to changes in size and can be used for dense prediction tasks without traditional dense layers. The authors  es in size and can be used for dense prediction tasks without traditional dense layers. The authors compare PVT to other state-of-the-art models, including LightCNN, and show that PVT outperforms them in terms of accuracy and robustness. Overall, the article presents a new and effective vision transformer The article discusses the importance of embracing failures as a natural part of the learning process. It highlights that by embracing uncertainty and taking on new challenges, individuals can develop confidence, self-esteem, and new skills. The article emphasizes the need to develop a positive and resilient attitude towards challenges, which is crucial for achieving personal growth and development. The text does not provide a clear problem or question, and the data is not suitable for analysis or visualization. The model's performance is not surprising, given the lack of * Developing a  lysis or visualization. The model's performance is not surprising, given the lack of * Developing a positive and resilient attitude towards challenges is key to achieving a more fulfilling and successful life.* Taking on new challenges can lead to significant personal and professional benefits.* Embracing uncertainty and taking on new challenges is essential for personal growth and development.* By doing so, individuals can build confidence, develop new skills, and expand their perspectives.* This mindset can also lead to increased creativity, productivity, and innovation.* By embracing challenges and uncertainty, individuals can unlock their full potential and
The paper proposes two new methods for training deep neural networks: Ensemble of Temporally Decorrelated Models (ETDM) and Deep Reinforcement Learning with Autoencoder (DRL-AE). ETDM combines multiple models with different architectures to capture complex behavior, while DRL-AE combines techniques from reinforcement learning and deep learning to optimize neural networks. The paper also explores the relation between the two methods and discusses potential applications in computer vision, natural language processing. The authors discuss challenges and trade-offs associated with The paper discussed the importance of scalability in datacenter design, emphasizing the need to understand trade-offs between different design elements such as power consumption, cooling, and capacity. The study concluded that scalability is a critical aspect of datacenter design and that understanding these trade- luded that scalability is a critical aspect of datacenter design and that understanding these trade-offs is essential for building efficient and cost-effective datacenters. The second part of the paper explored the potential of few-shot learning in computer vision tasks, proposing a new approach that involves pre-training a model on a large corpus of text data and The study evaluated the performance of a few-shot learning model on large-scale image classification tasks. The model achieved an accuracy of 71.4% on the ImageNet dataset, but its performance was lower than that of state-of-the-art models. The study suggests that few-shot models may not be effective for very large datasets and that additional research is needed to improve their performance. The model was able to predict the final score of 2-1 in favor of the home team with 100% accuracy The model was able to predict the final  l score of 2-1 in favor of the home team with 100% accuracy The model was able to predict the final score of 2-1 in favor of the home team, which was correct. The model was also able to identify the player who scored the winning goal and the player who committed the penalty that led to the away team's goal. This demonstrates the model's ability to analyze and understand the game, and to make accurate predictions based on the data it was trained on.Overall, the model's performance was impressive, and it suggests that the model is capable of